# EE3015 Sound Lab in Python
## One period of a real instrument: Fourier series and periodic extension with `kron`

This notebook is the Python (NumPy) version of the MATLAB sound lab of **Discussion 10** and of **Experiment 5** of the [Fourier Sound Lab](https://soheil-mohajer.github.io/Signals/sound-lab.html#instrument).
Run the cells from top to bottom (Shift + Enter). In [Google Colab](https://colab.research.google.com/github/soheil-mohajer/Signals/blob/main/python/sound_lab.ipynb) nothing needs to be installed.

**MATLAB → Python**

| MATLAB | Python (NumPy) |
|---|---|
| `x(1)` (first sample) | `x[0]`: Python counts from 0 |
| `x(a:b)` | `x[a-1:b]` |
| `fft(x)/N` | `np.fft.fft(x)/N` |
| `fftshift(a)` | `np.fft.fftshift(a)` |
| `kron(ones(M,1), v)` | `np.kron(np.ones(M), v)` (or `np.tile(v, M)`) |
| `circshift(v, s)` | `np.roll(v, s)` |
| `flipud(v)` | `v[::-1]` |
| `sound(x, fs)` | `play(x)` (defined below) |

In [ ]:
import io, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display

# The same recordings as in Musical_Instrument_Sound_Files.mat, saved as WAV files on the course website
BASE = "https://soheil-mohajer.github.io/Signals/audio/"

def load(name):
    """Download a recording and return (fs, x) with x as floats in [-1, 1]."""
    with urllib.request.urlopen(BASE + name + ".wav") as r:
        fs, x = wavfile.read(io.BytesIO(r.read()))
    return fs, x.astype(float) / 32768

def play(x):
    """Show an audio player for the signal x (like MATLAB's sound(x, fs))."""
    display(Audio(x, rate=fs))

fs, oboe = load("oboe")
_, sax = load("sax")
_, piano = load("piano")
print("fs =", fs, "Hz;  oboe:", len(oboe), "samples =", round(len(oboe) / fs, 2), "seconds")

## 1. Listen and look (parts (a) and (b))

In [ ]:
play(oboe)
t = np.arange(len(oboe)) / fs
plt.figure(figsize=(10, 3))
plt.plot(t, oboe)
plt.xlabel("t (seconds)"); plt.ylabel("Amplitude"); plt.title("Oboe")
plt.show()

Zoom in the middle of the note and cut out **exactly one period**. In MATLAB, the discussion uses `xstart = 29997` and `xend = 30096`, i.e., $N = 100$ samples.
Python counts from 0, so MATLAB's sample 29997 is `oboe[29996]`, and `oboe[a:b]` contains the samples `a, a+1, ..., b-1` ($b-a$ samples).

In [ ]:
xstart = 29997 - 1                 # MATLAB index 29997 -> Python index 29996
N = 100                            # one period has N = 100 samples
oboe_one_period = oboe[xstart:xstart + N]
print("f0 = fs/N =", fs / N, "Hz")

n = np.arange(xstart, xstart + 3 * N)
plt.figure(figsize=(10, 3))
plt.plot(n, oboe[n], label="recording (3 periods)")
plt.plot(n[:N], oboe_one_period, lw=3, label="oboe_one_period")
plt.xlabel("n"); plt.legend(); plt.show()

## 2. Fourier series coefficients (parts (c) and (d))

`np.fft.fft(v)/N` returns the DTFS coefficients $a_0, a_1, \dots, a_{N-1}$; `np.fft.fftshift` moves $a_0$ to the middle.

In [ ]:
ak = np.fft.fft(oboe_one_period) / N
k = np.arange(N) - N // 2                    # k = -N/2, ..., N/2 - 1
plt.figure(figsize=(10, 3))
plt.stem(k, np.abs(np.fft.fftshift(ak)))
plt.xlabel("k"); plt.ylabel("|a_k|"); plt.title("Oboe: FS coefficients")
plt.show()

## 3. Periodic extension with `kron` (part (k))

The Kronecker product `np.kron(np.ones(M), v)` puts `M` copies of `v` one after the other. If `v` is one period of a signal, this is $M$ periods of the periodic signal whose single period is `v`.

In [ ]:
print(np.kron(np.ones(3), np.array([1, 2, 3])))

**3.1 One period, repeated.** Build a 3-second signal `y1` from `oboe_one_period`.
How does it sound compared with the original recording? Without any computation: what are the FS coefficients of `y1`?

In [ ]:
M1 = round(3 * fs / N)                       # number of copies for 3 seconds
y1 = np.kron(np.ones(M1), oboe_one_period)
play(y1)

def show_copies(y, L, title):
    """Plot four copies of a block of length L, with the junctions marked."""
    n = np.arange(4 * L)
    plt.figure(figsize=(10, 3))
    plt.plot(n / fs * 1000, y[:4 * L])
    for m in range(1, 4):
        plt.axvline(m * L / fs * 1000, color="tab:orange", ls="--")
    plt.xlabel("t (ms)"); plt.title(title); plt.show()

show_copies(y1, N, "One period, repeated")

**3.2 One and a half periods, repeated.** Cut out $L = 1.5N = 150$ samples and repeat them.
Before you listen: what is the fundamental period of `y2` (in samples and in seconds), and what is its fundamental frequency? Does it sound like the same note?

In [ ]:
L = int(1.5 * N)                             # 150 samples
oboe_15 = oboe[xstart:xstart + L]
M2 = round(3 * fs / L)
y2 = np.kron(np.ones(M2), oboe_15)
play(y2)
show_copies(y2, L, "1.5 periods, repeated")

**3.3 Why?** Compare the FS coefficients of `y2` (computed from one period of `y2`, i.e., `oboe_15`) with those of one true period, on a frequency axis in Hz.
Which frequencies of the oboe survive, and which ones are new?

In [ ]:
ck = np.fft.fft(oboe_15) / L
plt.figure(figsize=(10, 4))
kk = np.arange(L // 2)
plt.stem(kk * fs / L, np.abs(ck[kk]), label="|c_k| of y2 (lines at k fs/L)")
kk = np.arange(N // 2)
plt.plot(kk * fs / N, np.abs(ak[kk]), "o", color="tab:orange", mfc="none", ms=10, label="|a_k| of one true period (lines at k fs/N)")
plt.xlim(-100, 6000); plt.xlabel("frequency (Hz)"); plt.legend(); plt.show()

**3.4 Two periods.** What would you hear if you repeated exactly *two* periods? Predict, then check.

In [ ]:
y3 = np.kron(np.ones(M1 // 2), oboe[xstart:xstart + 2 * N])
play(y3)

## 4. One time shift: an octave higher, or a clarinet (part (l))

Let $x[n]$ be the periodic oboe signal with period $N = 100$ and coefficients $a_k$, and let
$$z[n] = x[n] + x[n - N/2], \qquad w[n] = x[n] - x[n - N/2].$$
Before running the cell, use the time-shift property to find the FS coefficients of $z$ and $w$. Which one sounds one octave higher? Which one sounds like the odd-harmonic synthesis of part (j)?
(Within one period, a shift of the periodic signal is a *circular* shift: `np.roll`.)

In [ ]:
x = oboe_one_period
x_sh = np.roll(x, N // 2)                    # one period of x[n - N/2]
z = x + x_sh
w = x - x_sh
play(np.kron(np.ones(M1), z))
play(np.kron(np.ones(M1), w))

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].stem(np.arange(16), np.abs(np.fft.fft(z) / N)[:16]); ax[0].set_title("|FS coefficients| of z")
ax[1].stem(np.arange(16), np.abs(np.fft.fft(w) / N)[:16]); ax[1].set_title("|FS coefficients| of w")
ax[1].set_xlabel("k"); plt.show()

## 5. Playing it backwards (part (m))

Reverse one period and repeat it. The waveform looks different; does it sound different? Explain with the DTFS properties (and part (i)).
Then listen to the whole piano note backwards. Why does that one sound so different?

In [ ]:
v = oboe_one_period[::-1]
play(np.kron(np.ones(M1), v))
print("largest difference between |FFT| of v and of the original period:",
      np.max(np.abs(np.abs(np.fft.fft(v)) - np.abs(np.fft.fft(oboe_one_period)))))
play(piano[::-1])

## 6. Try it yourself

* Repeat Sections 3 to 5 for the saxophone and the piano. One period is about 105 samples for the saxophone (e.g., starting at MATLAB index 143914) and about 200 samples for the piano (e.g., starting at MATLAB index 31298).
* Repeat a block of $L = N - 1$ or $L = N + 1$ samples. What changes, and why is it hard to hear?
* Compare with Experiment 5 of the [Fourier Sound Lab](https://soheil-mohajer.github.io/Signals/sound-lab.html#instrument), where you can change $L$ with a slider.